# A1.12 · Cascading hallucination

**Function A — Securing AI Architectures → CyberTravels' Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.11 · Rogue agents in a multi-agent system](https://spbreed.github.io/cyber-commons/lessons/A1.11.html)**.

| | |
|---|---|
| Tools used | Inspect, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Let one fabricated fact travel three hops and watch its confidence rise as its provenance disappears.

**Why a security engineer needs it.** A single fabrication becomes a shared premise, and by the third hop nothing in the system records that it was ever uncertain. The control it builds is: verification against ground truth before a claim propagates (A3.5).

This is a **risk** lesson: it shows the failure happening before anything tries to stop it, so the control that follows is answering something you have already watched go wrong.

## 1 · The hook

Agent one is 90% accurate, which sounds fine. Agent two consumes its output as fact, and agent three consumes that. By the third hop the confident wrong answer has been repeated enough times that it reads like corroboration.

> **At CyberTravels.** The advisor is confident about a hotel that closed in 2024. The workflow agent books it, the file system agent validates an invoice against it, and the report to the executive cites three agreeing sources. R2.

## 2 · The framework

```
   agent 1        agent 2        agent 3        report
   90% right ---> takes as ----> takes as ----> "three sources agree"
                  fact           fact

   error compounds: 0.9 -> 0.81 -> 0.73
   confidence compounds the other way, because repetition reads as corroboration
```

**OWASP T5 — Cascading Hallucination Attacks. LLM09 — Misinformation.**

The **model** component produces confident text. Sometimes the text is wrong.
That is a known property, and on its own it is a quality problem rather than a
security one.

It becomes a security problem when the architecture has more than one step,
because an unverified claim from step one is an *input* to step two. And inputs
are not re-examined — that is the point of a pipeline.

Watch what happens to a single fabrication as it travels:

**Hop one.** "I could not find a CVE for this dependency, it is probably fine."
Hedged, and the hedge is visible.

**Hop two.** The next agent summarises: "dependency has no known CVEs."
The hedge is gone. Nothing lied — summarising removes qualifiers, that is what
summarising is.

**Hop three.** "Dependency verified clean." Now it is a finding, with the
confidence of something that was checked, and no field anywhere records that
nobody checked anything.

The security consequence is that **confidence rises as evidence disappears**,
which is exactly backwards. And it is not limited to accidents: an attacker who
can inject one plausible claim early gets it laundered into an established fact
by your own pipeline, which is why this sits in the threat taxonomy rather than
in a quality backlog.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

One hedged guess, three hops, and the confidence it acquires on the way.

## 4 · The check, as a skill

A hedged sentence about libfoo becomes a confident claim in three hops. The skill tracks both series — confidence and surviving provenance — because either one alone looks ordinary and the pair is the finding.

### The skill — [`skills/threats/confidence-provenance-decay-check/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/threats/confidence-provenance-decay-check/SKILL.md)

```yaml
name: confidence-provenance-decay-check
description: >-
  Track a hedged claim across summarisation hops and measure confidence rising
  while provenance disappears. Use when output from one model or agent becomes
  input to another, in report chains, research pipelines, or any place a summary
  is summarised.
allowed-tools: Read, Grep, Glob
```

# Confidence rises at exactly the rate evidence disappears

"I could not find a CVE, it is probably fine" becomes "libfoo is clean" in
three hops. Nothing in the chain lied. Each step did what summarising does —
dropped the hedge, dropped the caveat, dropped the sentence saying where the
claim came from — and the result is a confident statement with no evidence
behind it.

## When to use this

Report generation, triage pipelines, research chains, any agent that consumes
another agent's output, and any workflow where a human reads only the last
artefact.

## Procedure

**1 — Find the chains.** Every place where generated text is input to a later
generation step. Include the human-visible summary at the end; it is a hop.

**2 — Instrument the original claim.** Record its hedge, its confidence if one
is stated, and its provenance — the source it rests on.

**3 — Step the chain and record both series.** After each hop, the claim's
confidence and its surviving provenance fields. Two numbers per hop; the shape
is the finding.

**4 — Identify the hop where provenance empties.** That is where the claim
became unfalsifiable, and it is almost always earlier than where the confidence
peaked.

**5 — Test the carry rule.** A chain that propagates confidence and provenance
as structured fields, rather than as prose the next step must re-read, does not
decay this way. Check whether the interface has those fields at all.

## Example

**Input** — the fixture committed at the top of [`scripts/confidence_provenance_decay_check.py`](scripts/confidence_provenance_decay_check.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
 hop   confidence  claim
   0         0.20  I could not find a CVE for libfoo, it is probably fine
   1         0.80  a CVE for libfoo, it is fine
   2         0.80  a CVE for libfoo, it is fine
   3         0.80  a CVE for libfoo, it is fine

provenance recorded at hop 3: none
confidence at hop 0: 0.20   at hop 3: 0.8
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "chain": [{"hop": 0, "claim": "str", "confidence": 0.0, "provenance": ["str"]}],
  "confidence_delta": 0.0,
  "provenance_lost_at": 0,
  "interface": {"carries_confidence": false, "carries_provenance": false},
  "human_reads_hop": 0
}
```

## Failure modes

- **Judging the final claim on its own.** It reads well. That is the problem.
- **Measuring only confidence.** The pair is the finding; either alone is
  ordinary.
- **Fixing it with a prompt.** "Preserve caveats" is advisory; a field is not.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/threats/confidence-provenance-decay-check/scripts/confidence_provenance_decay_check.py
SCRIPT = "skills/threats/confidence-provenance-decay-check/scripts/confidence_provenance_decay_check.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan. Both, or the scanning skills clone successfully and then find
    # nothing to look at.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

A hedged guess at confidence 0.2 becomes a confident claim above 0.8 in three hops, while the provenance field empties — confidence rising at exactly the rate evidence disappears.

## Your turn

Take a finding your pipeline produced and try to walk it back to the step that first asserted it. If you cannot reach a step that checked something, you have found a cascade rather than a finding.

---

**Next → [A1.13 · Resource overload](https://spbreed.github.io/cyber-commons/lessons/A1.13.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.12.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.12.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*